# DenseNet-121 Experiments for ECG Heartbeat Classification

This notebook documents my DenseNet-121 experiments for a five-class ECG image classification project developed at Cornell Tech.

The task involves 87,554 labeled ECG images and a highly imbalanced class distribution, with more than 80% of the training samples belonging to the normal class (class 0). The test set also contains a simulated distribution shift representing ECG images collected from different hospitals or devices.

My work focused on exploring DenseNet-121 under different preprocessing and robustness strategies, including grayscale and RGB inputs, ECG-specific augmentation, Mixup, class-weighted loss, and test-time augmentation (TTA).

## Experiments

| Experiment | Main Configuration | Public Macro-F1 |
|---|---|---:|
| Experiment 1 | Grayscale DenseNet-121 + ECG-specific augmentation + Mixup + class-weighted loss + 5-view TTA | 0.72146 |
| Experiment 2 | Simplified grayscale DenseNet-121 baseline | 0.70882 |
| Experiment 3 | RGB DenseNet-121 + ImageNet normalization | 0.72812 |

**Evaluation Metric:** Macro-F1 was used because the dataset is severely imbalanced and performance on minority heartbeat classes is important.

> **Note:** This notebook is intended as a technical project showcase rather than a fully reproducible training package. Experiments 2 and 3 contain only the components that differ from the preceding experiment, while the original experimental results are preserved.

## Environment Setup

### Google Colab / Google Drive Setup

The original experiments were trained in Google Colab. The following cells mount Google Drive and extract the ECG image dataset used during development.

In [ ]:
import os
import zipfile

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
zip_path = '/content/drive/MyDrive/AML/aml-final-ecg-heart-beat-classification.zip'

In [ ]:
extract_root = '/content/temp_extract'
print(f"Extracting dataset from {zip_path} ...")

Extracting dataset from /content/drive/MyDrive/AML/aml-final-ecg-heart-beat-classification.zip ...


In [ ]:
if not os.path.exists(extract_root):
    os.makedirs(extract_root)

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_root)
    print("Dataset extracted successfully.")
else:
    print(f"Dataset archive not found: {zip_path}")
    print("Please verify the dataset path before running this notebook.")

Dataset extracted successfully.


In [ ]:
DATA_DIR = os.path.join(extract_root, 'ecg_image_data')

if os.path.exists(DATA_DIR):
    print(f"Dataset directory: {DATA_DIR}")
else:
    # Fallback in case the archive does not contain the expected top-level folder.
    DATA_DIR = extract_root
    print(f"Using extraction root as dataset directory: {DATA_DIR}")

print(f"Directory contents: {os.listdir(DATA_DIR)}")

Using extraction root as dataset directory: /content/temp_extract/ecg_image_data
Directory contents: ['sample_submission.csv', 'train_labels.csv', 'test', 'train']


# Experiment 1 — Robust Grayscale DenseNet-121

This experiment focuses on robustness to class imbalance and distribution shift.

The pipeline uses:
- grayscale ECG inputs,
- small rotations and translations,
- brightness and contrast augmentation,
- waveform thickening and color inversion,
- Mixup,
- inverse-square-root class weighting,
- and five-view test-time augmentation (TTA).

The goal is to reduce overfitting to the dominant normal class while improving robustness to variations in ECG image style.

In [ ]:
# import needed packages

In [ ]:
import os
import pandas as pd
import numpy as np

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from PIL import Image
from tqdm import tqdm

# --- 1. Configuration ---

In [ ]:
CONFIG = {
    'seed': 42,
    'batch_size': 32,
    'lr': 1e-4,
    'epochs': 25,
    'image_size': 224,
    'num_classes': 5,
    'num_workers': 2,
    'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    'data_dir': DATA_DIR
}

In [ ]:
print(f"Using device: {CONFIG['device']}")
print(f"Batch Size: {CONFIG['batch_size']}")

Using device: cuda
Batch Size: 32


In [ ]:
class ThickenLinesAndInvert(object):
    """
    ECG-specific preprocessing transform.

    1. Invert the grayscale image so that the ECG waveform becomes
       bright against a dark background.
    2. Apply max pooling as a simple dilation operation to thicken
       thin waveform strokes.

    The goal is to emphasize ECG waveform structures while reducing
    sensitivity to thin or noisy line rendering.
    """
    def __init__(self, kernel_size=3):
        self.pool = nn.MaxPool2d(kernel_size, stride=1, padding=kernel_size//2)

    def __call__(self, img_tensor):
        # Input shape: (C, H, W), with values in [0, 1].

        # Convert a white-background / dark-waveform image into
        # a dark-background / bright-waveform representation.
        img_inverted = 1.0 - img_tensor

        # MaxPool2d expects a batch dimension, so temporarily add one.
        # The operation acts similarly to morphological dilation.
        img_thick = self.pool(img_inverted.unsqueeze(0)).squeeze(0)

        return img_thick

# --- 2. Mixup Utilities ---

## Mixup Regularization

Mixup creates interpolated training examples and labels to reduce overconfidence and improve generalization. It was used in Experiment 1 as an additional regularization mechanism under severe class imbalance and distribution shift.

In [ ]:
def mixup_data(x, y, alpha=1.0, device='cuda'):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# --- 3. Dataset Definition ---

In [ ]:
class ECGDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, is_test=False):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        if self.is_test:
            img_name = f"{row['id']}.png"
            label = -1
        else:
            if 'filename' in row:
                img_name = row['filename']
            else:
                img_name = f"{row['id']}_{row['label']}.png"
            label = row['label']

        img_path = os.path.join(self.img_dir, img_name)

        try:
            image = Image.open(img_path).convert('L') # Convert ECG image to grayscale.
        except FileNotFoundError:
            alt_path = os.path.join(self.img_dir, f"{row['id']}.png")
            if os.path.exists(alt_path):
                 image = Image.open(alt_path).convert('L')
            else:
                raise FileNotFoundError(f"Image not found: {img_path}")

        if self.transform:
            image = self.transform(image)

        if self.is_test:
            return image, row['id']
        else:
            return image, torch.tensor(label, dtype=torch.long)

# --- 4. Data Augmentation Strategy ---

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])), # resize
    transforms.RandomRotation(degrees=10), # Small geometric perturbations simulate variations in ECG rendering.

    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),  # Mild intensity variation improves robustness to image-style changes.
    transforms.ColorJitter(brightness=0.1, contrast=0.1), 

    transforms.ToTensor(),

    ThickenLinesAndInvert(kernel_size=3),# ECG-specific waveform enhancement.
    transforms.Normalize(mean=[0.5], std=[0.5]) # normalize
])

In [ ]:
val_transforms = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.ToTensor(),

    ThickenLinesAndInvert(kernel_size=3),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# --- 5. Data Preparation ---

In [ ]:
csv_path = os.path.join(CONFIG['data_dir'], 'train_labels.csv')
if not os.path.exists(csv_path):
    csv_path = os.path.join(extract_root, 'train_labels.csv')

In [ ]:
full_df = pd.read_csv(csv_path)

In [ ]:
full_df['filename'] = full_df.apply(lambda x: f"{x['id']}_{x['label']}.png", axis=1)

In [ ]:
train_df, val_df = train_test_split(
    full_df, test_size=0.2, stratify=full_df['label'], random_state=CONFIG['seed']
) # Stratified 80/20 split preserves the original class distribution.

In [ ]:
train_img_dir = os.path.join(CONFIG['data_dir'], 'train')
train_dataset = ECGDataset(train_df, train_img_dir, transform=train_transforms)
val_dataset = ECGDataset(val_df, train_img_dir, transform=val_transforms)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=CONFIG['num_workers'], pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=True)

# --- 6. DenseNet-121 and Class Weighting ---

In [ ]:
class_counts = full_df['label'].value_counts().sort_index().values
# Smoothed inverse-frequency weighting:
# inverse square root reduces majority-class dominance without
# assigning excessively large weights to rare classes.

weights = 1.0 / np.sqrt(class_counts) 

weights = weights / weights.sum() * len(class_counts)
weights = torch.FloatTensor(weights).to(CONFIG['device'])
print(f"Smoothed Class Weights: {weights}")

Smoothed Class Weights: tensor([0.2063, 1.1778, 0.7299, 2.1934, 0.6925], device='cuda:0')


In [ ]:
print("Loading pretrained DenseNet-121 ...")
model = models.densenet121(pretrained=True)

Loading pretrained DenseNet-121 ...
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 30.8M/30.8M [00:00<00:00, 125MB/s]


In [ ]:
#灰階

In [ ]:
model.features.conv0 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

In [ ]:
num_ftrs = model.classifier.in_features
model.classifier = nn.Linear(num_ftrs, CONFIG['num_classes'])

In [ ]:
model = model.to(CONFIG['device'])

In [ ]:
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3)

In [ ]:
# --- 6.5 Verify Gpu ---

In [ ]:
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print("Using GPU")
else:
    print("Running on CPU")
    print("Wrong NVIDIA setting or wrong PyTorch version.")

PyTorch version: 2.9.0+cu126
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
Using GPU


# --- 7. Training Loop with Mixup (with Mixup) ---

### Training Procedure

The model was optimized using AdamW with a learning rate of `1e-4`. Model checkpoints were selected using validation Macro-F1 rather than overall accuracy because of the severe class imbalance.

The validation Macro-F1 reached a peak of approximately **0.929** during training. The substantially lower public leaderboard score illustrates the difficulty of generalizing to the shifted test distribution.

In [ ]:
best_f1 = 0.0

In [ ]:
for epoch in range(CONFIG['epochs']):
    print(f"\nEpoch {epoch+1}/{CONFIG['epochs']}")

    # Train
    model.train()
    train_loss = 0.0
    for images, labels in tqdm(train_loader, desc="Training"):
        images, labels = images.to(CONFIG['device']), labels.to(CONFIG['device'])

        optimizer.zero_grad()

        # Mixup
        images, labels_a, labels_b, lam = mixup_data(images, labels, alpha=1.0, device=CONFIG['device'])
        outputs = model(images)
        loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)

        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Validation", leave=False):
            images, labels = images.to(CONFIG['device']), labels.to(CONFIG['device'])
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    macro_f1 = f1_score(all_labels, all_preds, average='macro')

    print(f"Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val F1: {macro_f1:.4f}")

    scheduler.step(avg_val_loss)

    if macro_f1 > best_f1:
        best_f1 = macro_f1
        torch.save(model.state_dict(), "best_densenet.pth")
        print(">>> Saved Best Model (DenseNet-121)")
        print(classification_report(all_labels, all_preds, digits=4))


Epoch 1/25


Training: 100%|██████████| 2189/2189 [02:43<00:00, 13.41it/s]


Loss: 0.6216 | Val Loss: 0.2280 | Val F1: 0.8089
>>> Saved Best Model (DenseNet-121)
              precision    recall  f1-score   support

           0     0.9901    0.9623    0.9760     14494
           1     0.7137    0.7843    0.7473       445
           2     0.8953    0.9301    0.9123      1158
           3     0.3033    0.8672    0.4494       128
           4     0.9312    0.9891    0.9593      1286

    accuracy                         0.9569     17511
   macro avg     0.7667    0.9066    0.8089     17511
weighted avg     0.9674    0.9569    0.9609     17511


Epoch 2/25


Training: 100%|██████████| 2189/2189 [02:42<00:00, 13.49it/s]


Loss: 0.4780 | Val Loss: 0.3043 | Val F1: 0.7559

Epoch 3/25


...

Loss: 0.2570 | Val Loss: 0.0937 | Val F1: 0.9196

Epoch 24/25


Training: 100%|██████████| 2189/2189 [02:41<00:00, 13.57it/s]


Loss: 0.2542 | Val Loss: 0.0930 | Val F1: 0.9268

Epoch 25/25


Training: 100%|██████████| 2189/2189 [02:41<00:00, 13.53it/s]
                                                             

Loss: 0.2533 | Val Loss: 0.0931 | Val F1: 0.9195


# --- 8. Test-Time Augmentation (TTA) Inference ---

In [ ]:
# Perform five stochastic inference passes and aggregate
# class probabilities across views to reduce prediction variance.

In [ ]:
print("\n--- Starting TTA Inference ---")


--- Starting TTA Inference ---


In [ ]:
test_img_dir = os.path.join(CONFIG['data_dir'], 'test')
test_files = [f for f in os.listdir(test_img_dir) if f.endswith('.png')]
test_df = pd.DataFrame({'id': [f.split('.')[0] for f in test_files]})

In [ ]:
tta_transforms = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.RandomAffine(degrees=0, translate=(0.02, 0.02)),
    transforms.ToTensor(),
    ThickenLinesAndInvert(kernel_size=3), # ECG-specific preprocessing
    transforms.Normalize(mean=[0.5], std=[0.5])
])

In [ ]:
model.load_state_dict(torch.load("best_densenet.pth"))
model.eval()

DenseNet(
  (features): Sequential(
    (conv0): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu0): ReLU(inplace=True)
    (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (denseblock1): _DenseBlock(
      (denselayer1): _DenseLayer(
        (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu1): ReLU(inplace=True)
        (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu2): ReLU(inplace=True)
        (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      )
      (denselayer2): _DenseLayer(
        (norm1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu

In [ ]:
test_dataset_tta = ECGDataset(test_df, test_img_dir, transform=tta_transforms, is_test=True)
test_loader_tta = DataLoader(test_dataset_tta, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)

In [ ]:
final_predictions = {}
TTA_TIMES = 5

In [ ]:
for round_i in range(TTA_TIMES):
    print(f"TTA Round {round_i+1}/{TTA_TIMES}...")
    with torch.no_grad():
        for images, ids in tqdm(test_loader_tta, leave=False):
            images = images.to(CONFIG['device'])
            outputs = torch.softmax(model(images), dim=1)

            for id_val, prob in zip(ids, outputs.cpu().numpy()):
                if id_val not in final_predictions:
                    final_predictions[id_val] = prob
                else:
                    final_predictions[id_val] += prob

TTA Round 1/5...


TTA Round 2/5...


TTA Round 3/5...


TTA Round 4/5...


TTA Round 5/5...


In [ ]:
submission_data = []

In [ ]:
for id_val, total_prob in final_predictions.items():
    pred_label = np.argmax(total_prob)
    submission_data.append({'id': id_val, 'label': pred_label})

In [ ]:
sub_df = pd.DataFrame(submission_data)
sub_df.to_csv("submission_densenet_thick.csv", index=False)
print("✅ Submission file saved as 'submission_densenet_thick.csv'")

✅ Submission file saved as 'submission_densenet_thick.csv'


### Experiment 1 Result

**Public Kaggle Macro-F1: 0.72146**

The robustness-focused DenseNet substantially exceeded the from-scratch CNN baseline range observed in the project. Validation performance was considerably higher than public-test performance, indicating a meaningful train/test distribution shift.

# Experiment 2 — Simplified Grayscale DenseNet-121

To isolate the effect of the additional robustness techniques used in Experiment 1, this experiment removes most augmentation, Mixup, class weighting, and test-time augmentation.

Only the components that differ from Experiment 1 are shown below.

### Key Changes

- Image size: 224 → 256
- Training epochs: 25 → 20
- Removed geometric and intensity augmentation
- Removed Mixup
- Removed class-weighted loss
- Removed test-time augmentation

In [ ]:
CONFIG = {
    'seed': 42,
    'batch_size': 32,
    'lr': 1e-4,
    'epochs': 20, # Reduced from 25
    'image_size': 256, # Increased from 224
    'num_classes': 5,
    'num_workers': 2,
    'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    'data_dir': DATA_DIR
}

In [ ]:
# Remove the additional augmentation, Mixup, and TTA used in Experiment 1.

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

In [ ]:
val_transforms = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

In [ ]:
# Standard cross-entropy loss without class weighting.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=CONFIG['lr'])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3)

In [ ]:
# --- 7. Inference ---

In [ ]:
# Standard single-view inference without TTA.

In [ ]:
test_img_dir = os.path.join(CONFIG['data_dir'], 'test')
test_files = [f for f in os.listdir(test_img_dir) if f.endswith('.png')]
test_df = pd.DataFrame({'id': [f.split('.')[0] for f in test_files]})

In [ ]:
model.load_state_dict(torch.load("best_densenet_simple.pth"))
model.eval()

In [ ]:
test_dataset = ECGDataset(test_df, test_img_dir, transform=val_transforms, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)

In [ ]:
submission = []
with torch.no_grad():
    for images, ids in tqdm(test_loader):
        images = images.to(CONFIG['device'])
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        for id_val, pred in zip(ids, preds.cpu().numpy()):
            submission.append({'id': id_val, 'label': pred})

In [ ]:
pd.DataFrame(submission).to_csv("submission_densenet_simple.csv", index=False)
print("✅ submission_densenet_simple.csv created.")

### Experiment 2 Result

**Public Kaggle Macro-F1: 0.70882**

Removing the additional robustness mechanisms reduced public Macro-F1 relative to Experiment 1, suggesting that augmentation and imbalance-aware training provided useful generalization benefits.

# Experiment 3 — RGB DenseNet-121 with ImageNet Normalization

This experiment starts from the simplified DenseNet configuration in Experiment 2 and changes the input representation from grayscale to RGB.

The purpose is to better align the input format with the ImageNet-pretrained DenseNet-121 backbone.

### Key Changes

- Grayscale input → 3-channel RGB input
- Batch size: 32 → 64
- ImageNet normalization instead of generic grayscale normalization
- Standard DenseNet-121 input convolution is retained
- No class weighting
- No test-time augmentation

Only the components that differ from the previous experiment are shown below.

In [ ]:
CONFIG = {
    'seed': 42,
    'batch_size': 64,
    'lr': 1e-4,
    'epochs': 20,
    'image_size': 256,
    'num_classes': 5,
    'num_workers': 2,
    'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    'data_dir': DATA_DIR
}

In [ ]:
# Use ImageNet channel statistics to match the pretrained RGB backbone.

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
val_transforms = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# Keep the original 3-channel DenseNet input convolution for RGB images.

In [ ]:
print("Loading DenseNet-121...")
model = models.densenet121(pretrained=True)

In [ ]:
num_ftrs = model.classifier.in_features
model.classifier = nn.Linear(num_ftrs, CONFIG['num_classes'])

In [ ]:
# Standard cross-entropy loss without class weighting.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=CONFIG['lr'])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3)

In [ ]:
# --- 7. Inference ---

In [ ]:
# Standard single-view inference without TTA.

In [ ]:
test_img_dir = os.path.join(CONFIG['data_dir'], 'test')
test_files = [f for f in os.listdir(test_img_dir) if f.endswith('.png')]
test_df = pd.DataFrame({'id': [f.split('.')[0] for f in test_files]})

In [ ]:
model.load_state_dict(torch.load("best_densenet_rgb.pth"))
model.eval()

In [ ]:
test_dataset = ECGDataset(test_df, test_img_dir, transform=val_transforms, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)

In [ ]:
submission = []
with torch.no_grad():
    for images, ids in tqdm(test_loader):
        images = images.to(CONFIG['device'])
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        for id_val, pred in zip(ids, preds.cpu().numpy()):
            submission.append({'id': id_val, 'label': pred})

In [ ]:
pd.DataFrame(submission).to_csv("submission_densenet_rgb.csv", index=False)
print("✅ submission_densenet_rgb.csv created.")

### Experiment 3 Result

**Public Kaggle Macro-F1: 0.72812**

This was the highest public Macro-F1 recorded among the three DenseNet variants documented in this notebook.

## DenseNet Experiment Summary

The three experiments highlight the trade-off between explicit robustness mechanisms and closer alignment with the pretrained DenseNet backbone.

Experiment 1 introduced the most aggressive robustness strategy through ECG-specific augmentation, Mixup, class weighting, and TTA, achieving a public Macro-F1 of **0.72146**.

Experiment 2 removed these mechanisms and dropped to **0.70882**, providing evidence that the additional robustness techniques were beneficial.

Experiment 3 used RGB inputs and ImageNet normalization and achieved the highest score recorded in this notebook at **0.72812**.

Overall, the experiments suggest that both robustness-oriented preprocessing and the input representation of pretrained models can materially affect generalization under distribution shift.